# rfz.go.kr (menuno=149) 크롤러 v17 — 선행 인덱스(숫자/범위) 안전 제거

앞머리에 `194~195.` 같은 **범위형 인덱스**도 있어 단순 1~3자리 제거로는 부족했습니다.

이 버전은 제목의 **맨 앞**에서 다음 패턴을 한 덩어리로 잘라냅니다:
- `숫자+` (자릿수 무제한)
- 선택적으로 `~`, `-`, `–`, `—` 뒤에 또 **숫자+`**(범위)
- 선택적으로 구분기호(`. ) - : ㆍ ·`)
- 그리고 공백들

예) `1 제목`, `09. 제목`, `100) 제목`, `194~195. 제목`, `12-13 제목` → 앞 인덱스 제거 후 `제목`만 남김.

본문 중의 `제135조`, `제1항` 등은 그대로 보존됩니다.


In [ ]:
!pip install -q selenium webdriver-manager beautifulsoup4 lxml


In [3]:
import re, csv, time
from time import monotonic
from typing import List, Tuple, Optional
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager

URL = "https://rfz.go.kr/index.html?menuno=149"
OUTPUT_CSV = "rfz_menuno149_allopts_page1_2.csv"
WAIT_SEC = 60
MAX_PAGES = 2
UL_XPATH = "/html/body/div[3]/div[2]/form/ul"
FORM_XPATH = "/html/body/div[3]/div[2]/form"
DROPDOWN_XPATH = "/html/body/div[3]/div[2]/form/div[2]/span[1]/select"
APPLY_BTN_XPATH = "/html/body/div[3]/div[2]/form/div[2]/button"

def setup_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--lang=ko-KR")
    opts.page_load_strategy = "eager"
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    driver.set_page_load_timeout(90)
    return driver

def strip_specials_keep_kr_en_num_space(s: str) -> str:
    if s is None: return ""
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\uAC00-\uD7A3A-Za-z0-9 ]+", "", s)
    return re.sub(r"\s{2,}", " ", s).strip()

# 선행 인덱스(숫자 또는 숫자~숫자/숫자-숫자) + 선택적 구분기호 + 공백 제거
LEADING_INDEX_RE = re.compile(r"^\s*\d+(?:\s*[~\-–—]\s*\d+)?\s*[\.)\-:ㆍ·]?\s*")
def drop_leading_index_any(s: str) -> str:
    if not s:
        return ""
    return LEADING_INDEX_RE.sub("", s)

BULLETS = "ㅇo○●•∙·"
def _heading_regex(words: List[str]) -> re.Pattern:
    joined = r"\s*".join(map(re.escape, words))
    pattern = rf"(?ms)(?:^|\n)\s*[{BULLETS}]?\s*{joined}\s*(?:[:：-])?\s*"
    return re.compile(pattern)

REL_HEADING = _heading_regex(["관계", "법령"])  # '관계법령'
SPEC_HEADING = _heading_regex(["규제", "특례", "사항"])  # '규제특례사항'

def parse_pre_by_headings(pre_html: str) -> Tuple[str, str]:
    if not pre_html:
        return "", ""
    soup = BeautifulSoup(pre_html, "lxml")
    pre = soup.find("pre") or soup
    text = pre.get_text("\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    m_rel = REL_HEADING.search(text)
    m_spec = SPEC_HEADING.search(text)
    if not m_rel and not m_spec:
        return "", ""
    rel_start = m_rel.end() if m_rel else None
    spec_start = m_spec.end() if m_spec else None
    if m_rel and m_spec:
        if m_rel.start() < m_spec.start():
            relaw = text[rel_start:m_spec.start()]
            spec = text[spec_start:]
        else:
            spec = text[spec_start:m_rel.start()]
            relaw = text[rel_start:]
    elif m_rel and not m_spec:
        relaw = text[rel_start:]; spec = ""
    else:
        relaw = ""; spec = text[spec_start:]
    relaw = re.sub(r"\s+", " ", relaw).strip()
    spec  = re.sub(r"\s+", " ", spec).strip()
    return relaw, spec

def parse_pre_by_strong_fallback(pre_html: str) -> Tuple[str, str]:
    if not pre_html:
        return "", ""
    soup = BeautifulSoup(pre_html, "lxml")
    pre = soup.find("pre") or soup
    s = pre.find_all("strong")
    if not s:
        return pre.get_text(" ", strip=True), ""
    cpy = BeautifulSoup(str(pre), "lxml")
    pre2 = cpy.find("pre") or cpy
    s2 = pre2.find_all("strong")
    if len(s2) >= 1: s2[0].string = "___RELSTART___"
    if len(s2) >= 2: s2[1].string = "___SPECSTART___"
    text = pre2.get_text(" ", strip=True)
    relaw, spec = "", ""
    if "___RELSTART___" in text:
        after_rel = text.split("___RELSTART___", 1)[1]
        if "___SPECSTART___" in after_rel:
            relaw, spec = after_rel.split("___SPECSTART___", 1)
        else:
            relaw = after_rel
    relaw = re.sub(r"\s+", " ", relaw).strip()
    spec  = re.sub(r"\s+", " ", spec).strip()
    return relaw, spec

def parse_pre_raw(pre_html: str) -> Tuple[str, str]:
    relaw, spec = parse_pre_by_headings(pre_html)
    if relaw or spec:
        return relaw, spec
    return parse_pre_by_strong_fallback(pre_html)

def list_signature(driver) -> str:
    anchors = driver.find_elements(By.XPATH, UL_XPATH + "/li/a")
    texts = []
    for a in anchors[:5]:
        try:
            if a.is_displayed():
                texts.append((a.text or "").strip())
        except StaleElementReferenceException:
            continue
    return "|".join(texts)

def first_anchor_and_text(driver):
    anchors = driver.find_elements(By.XPATH, UL_XPATH + "//li/a")
    for a in anchors:
        try:
            if a.is_displayed():
                t = (a.text or "").strip()
                if t:
                    return a, t
        except StaleElementReferenceException:
            continue
    return None, None

def wait_for_list_refresh(driver, prev_signature: Optional[str]) -> None:
    start = monotonic()
    try:
        prev_anchor, _ = first_anchor_and_text(driver)
        if prev_anchor is not None:
            WebDriverWait(driver, 20).until(EC.staleness_of(prev_anchor))
    except Exception:
        pass
    while monotonic() - start < WAIT_SEC:
        cur_sig = list_signature(driver)
        if cur_sig and (prev_signature is None or cur_sig != prev_signature):
            break
        time.sleep(0.35)
    stable_for, last_count = 0.0, -1
    while monotonic() - start < WAIT_SEC:
        count = len(driver.find_elements(By.XPATH, UL_XPATH + "/li/a"))
        if count == last_count and count > 0:
            stable_for += 0.25
            if stable_for >= 2.0:
                break
        else:
            stable_for, last_count = 0.0, count
        time.sleep(0.25)

def verify_selected_option(driver, desired_value: str, desired_text: str) -> bool:
    sel = driver.find_element(By.XPATH, DROPDOWN_XPATH)
    cur_val = driver.execute_script("return arguments[0].value;", sel)
    cur_text = driver.execute_script("return arguments[0].options[arguments[0].selectedIndex].text;", sel)
    return (str(cur_val).strip() == str(desired_value).strip()) or (str(cur_text).strip() == str(desired_text).strip())

def robust_select_change(driver, idx: int) -> bool:
    for attempt in range(6):
        try:
            sel = WebDriverWait(driver, WAIT_SEC).until(EC.presence_of_element_located((By.XPATH, DROPDOWN_XPATH)))
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", sel)
            time.sleep(0.2)
            options = sel.find_elements(By.TAG_NAME, 'option')
            if idx >= len(options):
                return False
            desired_value = options[idx].get_attribute('value')
            desired_text  = (options[idx].text or '').strip()
            # 1) Selenium Select
            try:
                Select(sel).select_by_index(idx)
            except Exception:
                pass
            time.sleep(0.2)
            if verify_selected_option(driver, desired_value, desired_text):
                return True
            # 2) option 직접 클릭
            try:
                ActionChains(driver).move_to_element(sel).click().perform()
                time.sleep(0.1)
                ActionChains(driver).move_to_element(options[idx]).click().perform()
            except Exception:
                pass
            time.sleep(0.2)
            if verify_selected_option(driver, desired_value, desired_text):
                return True
            # 3) JS 설정 + 이벤트
            driver.execute_script(
                "arguments[0].selectedIndex = arguments[1];"
                "arguments[0].value = arguments[0].options[arguments[1]].value;"
                "arguments[0].dispatchEvent(new Event('input', {bubbles:true}));"
                "arguments[0].dispatchEvent(new Event('change', {bubbles:true}));",
                sel, idx
            )
            time.sleep(0.3)
            if verify_selected_option(driver, desired_value, desired_text):
                return True
            # 4) jQuery trigger
            try:
                driver.execute_script("if (window.jQuery) { jQuery(arguments[0]).val(arguments[1]).trigger('change'); }", sel, desired_value)
                time.sleep(0.3)
                if verify_selected_option(driver, desired_value, desired_text):
                    return True
            except Exception:
                pass
        except (StaleElementReferenceException, TimeoutException):
            time.sleep(0.8)
            continue
    return False

def click_apply_and_wait(driver, prev_signature: Optional[str]) -> None:
    try:
        btn = WebDriverWait(driver, WAIT_SEC).until(EC.element_to_be_clickable((By.XPATH, APPLY_BTN_XPATH)))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        time.sleep(0.1)
        try:
            btn.click()
        except Exception:
            driver.execute_script("arguments[0].click();", btn)
    except Exception:
        try:
            form = driver.find_element(By.XPATH, FORM_XPATH)
            driver.execute_script("if (arguments[0].requestSubmit) arguments[0].requestSubmit(); else arguments[0].submit();", form)
        except Exception:
            pass
    wait_for_list_refresh(driver, prev_signature)

def find_list_items(driver):
    WebDriverWait(driver, WAIT_SEC).until(EC.presence_of_element_located((By.XPATH, UL_XPATH)))
    li_xpath = UL_XPATH + "/li"
    items = driver.find_elements(By.XPATH, li_xpath)
    return li_xpath, items

def first_visible_nonempty_pre_html(driver) -> Optional[str]:
    pre_candidates = driver.find_elements(By.XPATH, UL_XPATH + "//pre")
    for _ in range(20):
        for p in pre_candidates:
            try:
                if p.is_displayed():
                    html = p.get_attribute("innerHTML") or ""
                    if BeautifulSoup(html, "lxml").get_text(strip=True):
                        return html
            except StaleElementReferenceException:
                continue
        time.sleep(0.5)
    return None

def get_active_pre_html(driver, li_xpath: str, idx: int) -> str:
    try:
        target_pre = driver.find_element(By.XPATH, f"{li_xpath}[{idx}]/div/pre")
        for _ in range(20):
            if target_pre.is_displayed():
                html = target_pre.get_attribute("innerHTML") or ""
                if BeautifulSoup(html, "lxml").get_text(strip=True):
                    return html
            time.sleep(0.5)
    except Exception:
        pass
    for cls in ["on", "active", "current", "selected"]:
        try:
            p = driver.find_element(By.XPATH, f"{UL_XPATH}/li[contains(@class,'{cls}')]/div/pre")
            if p.is_displayed():
                html = p.get_attribute("innerHTML") or ""
                if BeautifulSoup(html, "lxml").get_text(strip=True):
                    return html
        except Exception:
            continue
    html = first_visible_nonempty_pre_html(driver)
    if html:
        return html
    try:
        fallback = driver.find_element(By.XPATH, f"{UL_XPATH}/li[1]/div/pre")
        return fallback.get_attribute("innerHTML") or ""
    except Exception:
        return ""

def click_each_item_and_extract(driver):
    li_xpath, items = find_list_items(driver)
    rows = []
    N = len(items)
    for idx in range(1, N + 1):
        a_xpath = f"{li_xpath}[{idx}]/a"
        for attempt in range(6):
            try:
                a_el = WebDriverWait(driver, WAIT_SEC).until(EC.element_to_be_clickable((By.XPATH, a_xpath)))
                title_raw = (a_el.text or "").strip()
                driver.execute_script("arguments[0].click();", a_el)
                pre_html = get_active_pre_html(driver, li_xpath, idx)
                relaw_raw, spec_raw = parse_pre_raw(pre_html)
                rows.append({
                    "특례법령_raw": title_raw,
                    "관계법령_raw": relaw_raw,
                    "규제특례사항_raw": spec_raw,
                })
                break
            except (StaleElementReferenceException, TimeoutException):
                if attempt == 5:
                    pass
                else:
                    time.sleep(0.6)
                    continue
    return rows

def go_to_page(driver, page_num: int) -> bool:
    prev_sig = list_signature(driver)
    candidates = [
        f"//a[normalize-space(text())='{page_num}']",
        f"//button[normalize-space(text())='{page_num}']",
        f"//li/a[normalize-space(text())='{page_num}']",
    ]
    clicked = False
    for xp in candidates:
        try:
            el = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, xp)))
            driver.execute_script("arguments[0].click();", el)
            clicked = True
            break
        except Exception:
            continue
    if not clicked:
        return False
    wait_for_list_refresh(driver, prev_signature=prev_sig)
    return True

def count_visible_items(driver) -> int:
    li_xpath, items = find_list_items(driver)
    return len(driver.find_elements(By.XPATH, li_xpath + "/a"))

def select_all_options_and_crawl(headless=True):
    driver = setup_driver(headless=headless)
    rows_all = []
    try:
        driver.get(URL)
        WebDriverWait(driver, WAIT_SEC).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        sel_el = WebDriverWait(driver, WAIT_SEC).until(EC.presence_of_element_located((By.XPATH, DROPDOWN_XPATH)))
        initial_options = Select(sel_el).options
        initial_count = len(initial_options)
        print(f"드롭다운 옵션 수: {initial_count}")
        for opt_index in range(initial_count):
            try:
                go_to_page(driver, 1)
            except Exception:
                pass
            selected = robust_select_change(driver, opt_index)
            if not selected:
                print(f"[스킵] 옵션 {opt_index+1} 선택 실패")
                continue
            prev_sig = list_signature(driver)
            click_apply_and_wait(driver, prev_signature=prev_sig)
            try:
                c1 = count_visible_items(driver)
                print(f"옵션 {opt_index+1}, 페이지 1: {c1}건")
            except Exception:
                pass
            for page in range(1, MAX_PAGES + 1):
                if page > 1 and not go_to_page(driver, page):
                    print(f"옵션 {opt_index+1}, 페이지 {page}: 없음")
                    break
                try:
                    part_rows = click_each_item_and_extract(driver)
                    rows_all.extend(part_rows)
                    print(f"옵션 {opt_index+1}, 페이지 {page}: {len(part_rows)}건")
                except Exception as e:
                    print(f"[경고] 옵션 {opt_index+1}, 페이지 {page}: {e}")
                    continue
        return rows_all
    finally:
        driver.quit()

def save_csv(rows):
    with open(OUTPUT_CSV, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["특례법령", "관계법령", "규제특례사항"])
        w.writeheader()
        for r in rows:
            # 1) 맨 앞 숫자/범위 인덱스 제거 (예: '194~195.', '12-13', '07.' 등)
            title_no_index = drop_leading_index_any(r.get("특례법령_raw", ""))
            # 2) 특수문자만 제거(숫자 보존)
            title = strip_specials_keep_kr_en_num_space(title_no_index)
            relaw = strip_specials_keep_kr_en_num_space(r.get("관계법령_raw", ""))
            spec  = strip_specials_keep_kr_en_num_space(r.get("규제특례사항_raw", ""))
            w.writerow({
                "특례법령": title,
                "관계법령": relaw,
                "규제특례사항": spec,
            })
    print(f"[완료] {len(rows)}건 저장 → {OUTPUT_CSV}")


In [4]:
# ▶ 실행
rows = select_all_options_and_crawl(headless=True)
save_csv(rows)


드롭다운 옵션 수: 4
옵션 1, 페이지 1: 50건
옵션 1, 페이지 1: 50건
옵션 1, 페이지 2: 18건
옵션 2, 페이지 1: 13건
옵션 2, 페이지 1: 13건
옵션 2, 페이지 2: 없음
옵션 3, 페이지 1: 50건
옵션 3, 페이지 1: 50건
옵션 3, 페이지 2: 11건
옵션 4, 페이지 1: 32건
옵션 4, 페이지 1: 32건
옵션 4, 페이지 2: 없음
[완료] 174건 저장 → rfz_menuno149_allopts_page1_2.csv
